# Adaptive v2 Kaggle GPU Run

This notebook pulls the versioned `adaptive_v2` code, verifies the Kaggle GPU, runs one seed for 20 Qwen rounds, and saves a local output log and JSON results with family-level metrics.

In [6]:
import json
import os
import subprocess
import sys
from pathlib import Path

project_dir = Path("/kaggle/working/mastercard_hackathon")
repository_url = "https://github.com/keshav-0210/mastercard_hackathon.git"
if not project_dir.exists():
    subprocess.run(["git", "clone", repository_url, str(project_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)

for module_name in list(sys.modules):
    if module_name == "mastercard_defence" or module_name.startswith("mastercard_defence."):
        del sys.modules[module_name]

model_dir = Path("/kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local")
gguf_files = sorted(model_dir.glob("*.gguf"))
if len(gguf_files) < 2:
    raise FileNotFoundError(f"Expected two Qwen GGUF shards under {model_dir}, found: {gguf_files}")
model_path = next(path for path in gguf_files if "00001-of-00002" in path.name)

sys.path.insert(0, str(project_dir / "src"))
os.chdir(project_dir)
os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)
os.environ["GPU_LAYERS"] = "-1"

import torch
print("Project:", project_dir)
print("Model shards:", len(gguf_files))
print("Model path:", model_path)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "Enable GPU in Kaggle before running this notebook"
print("Kaggle GPU check: OK")
print("Qwen model loading is deferred until first inference call; QwenAgents will be used for this run")

Updating 22c906d..6d76b62
Fast-forward
 adaptive/adaptive.ipynb | 191 ++++++++++++++++++++++++++++++++----------------
 1 file changed, 130 insertions(+), 61 deletions(-)
Project: /kaggle/working/mastercard_hackathon
Model shards: 2
Model path: /kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
CUDA available: True
GPU: Tesla T4
Kaggle GPU check: OK
Qwen model loading is deferred until first inference call; QwenAgents will be used for this run


From https://github.com/keshav-0210/mastercard_hackathon
   22c906d..6d76b62  main       -> origin/main


In [2]:
%pip install --no-cache-dir ctgan

import ctgan
print("CTGAN version:", getattr(ctgan, "__version__", "installed"))
print("CTGAN installation: OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 51.6 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.
CTGAN version: 0.12.1
CTGAN installation: OK


In [3]:
%pip install --no-cache-dir --prefer-binary --index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --extra-index-url https://pypi.org/simple llama-cpp-python

from llama_cpp import llama_supports_gpu_offload
llama_gpu_available = bool(llama_supports_gpu_offload())
print("llama.cpp GPU offload support:", llama_gpu_available)
if not llama_gpu_available:
    print("Qwen GPU offload unavailable; CTGAN GPU will still be used and the smoke/full run will use HeuristicAgents.")
else:
    print("llama.cpp CUDA installation: OK")

Looking in indexes: https://abetlen.github.io/llama-cpp-python/whl/cu124, https://pypi.org/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 47.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


ggml_cuda_init: found 2 CUDA devices (Total VRAM: 29823 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB
  Device 1: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB


llama.cpp GPU offload support: True
llama.cpp CUDA installation: OK


In [7]:
import tempfile
from datetime import datetime, timezone

from mastercard_defence.agents import QwenAgents
from mastercard_defence.llm import SharedLocalLLM
from mastercard_defence.loop import ClosedLoop, load_config

RUN_ROUNDS = 20
QWEN_CONTEXT_SIZE = 4096
QWEN_MAX_OUTPUT_TOKENS = 256

config = load_config(str(project_dir / "config" / "default.yaml"))
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
config["paths"]["memory_db"] = str(project_dir / "artifacts" / f"adaptive_v2_memory_{run_stamp}.sqlite")
config["generator_backend"] = "ctgan"
# 3 epochs was far too few for CTGAN to converge: it produced near-garbage amount/velocity
# distributions that saturated every fidelity penalty term (behavioural_plausibility pinned at
# exactly 0.0) and destabilized the detector. CTGAN is fit once per suite, not per round, so a
# higher epoch count is a modest one-time cost, not a per-round cost.
config["generator_epochs"] = 50
config["generator_training_attack_size"] = 40
config["generator_training_reference_size"] = 400
config["detector_mode"] = "continual"
config["detector_retrain_every"] = 2
config["max_historical_pool"] = 800
config["max_replay_examples"] = 800
config["weakness_weight_multiplier"] = 8.0
config["model"]["context_size"] = QWEN_CONTEXT_SIZE
config["model"]["max_output_tokens"] = QWEN_MAX_OUTPUT_TOKENS
config["pipeline"]["rounds"] = RUN_ROUNDS
config["pipeline"]["synthetic_transactions"] = 400
config["pipeline"]["max_generated_attacks"] = 80
config["pipeline"]["fraud_rate"] = 0.02

agents = QwenAgents(config, llm=SharedLocalLLM(config))
loop = ClosedLoop(config, agents=agents)
print("Adaptive v2 configuration ready")
print("Agent backend:", type(agents).__name__)
print("Generator:", config["generator_backend"])
print("Generator epochs:", config["generator_epochs"])
print("Detector:", config["detector_mode"])
print("Detector retrain every:", config["detector_retrain_every"])
print("Max replay examples:", config["max_replay_examples"])
print("Weakness weight multiplier:", config["weakness_weight_multiplier"])
print("Target fraud rate:", config["pipeline"]["fraud_rate"])
print("Qwen context size:", config["model"]["context_size"])
print("Qwen max output tokens:", config["model"]["max_output_tokens"])
print("Seeds: 1")
print("Rounds:", RUN_ROUNDS)
print("Run timestamp:", run_stamp)


Adaptive v2 configuration ready
Agent backend: QwenAgents
Generator: ctgan
Generator epochs: 50
Detector: continual
Detector retrain every: 2
Max replay examples: 800
Weakness weight multiplier: 8.0
Target fraud rate: 0.02
Qwen context size: 4096
Qwen max output tokens: 256
Seeds: 1
Rounds: 20
Run timestamp: 20260828T194436Z


In [15]:
import tempfile

smoke_config = load_config(str(project_dir / "config" / "default.yaml"))
smoke_config["paths"]["memory_db"] = tempfile.mktemp(suffix="_adaptive_v2_smoke.sqlite")
smoke_config["generator_backend"] = "procedural"
smoke_config["detector_mode"] = "continual"
smoke_config["detector_retrain_every"] = 3
smoke_config["pipeline"]["rounds"] = 3
smoke_config["pipeline"]["synthetic_transactions"] = 400
smoke_config["pipeline"]["max_generated_attacks"] = 10
smoke_config["pipeline"]["fraud_rate"] = 0.02

# Reuse the single already-created `agents` (and its underlying loaded GGUF model) instead of
# instantiating a second SharedLocalLLM; loading two llama.cpp models onto the same dual-GPU
# session causes severe VRAM contention and can appear as an indefinite hang on the next load.
smoke_loop = ClosedLoop(smoke_config, agents=agents)
try:
    smoke_results = smoke_loop.run(rounds=3, seed=20260821)
finally:
    smoke_loop.close()

assert len(smoke_results) == 3
assert [result["round"] for result in smoke_results] == [1, 2, 3]
smoke_families = [result["specification"].attack_family for result in smoke_results]
assert all(set(result["detection"]["by_attack_family"]) == {family} for result, family in zip(smoke_results, smoke_families))
assert all(0.01 <= result["detection"]["training_fraud_rate"] <= 0.03 for result in smoke_results)
assert all(0.01 <= result["detection"]["validation_fraud_rate"] <= 0.03 for result in smoke_results)
assert all("detector_version" in result["detection"] for result in smoke_results)
assert all("historical_robustness" in result["detection"] for result in smoke_results)
coverage_progress = [result["diversity"]["cumulative_family_coverage_ratio"] for result in smoke_results]
assert coverage_progress == sorted(coverage_progress)
print("ADAPTIVE_V2_THREE_ROUND_SMOKE_OK")
print("Smoke rounds:", len(smoke_results))
print("Smoke families:", smoke_families)
print("Detector versions:", [result["detection"]["detector_version"] for result in smoke_results])
print("Cumulative family coverage:", coverage_progress)
print("Training fraud rates:", [result["detection"]["training_fraud_rate"] for result in smoke_results])
print("Validation fraud rates:", [result["detection"]["validation_fraud_rate"] for result in smoke_results])

[seed=20260821] starting 3-round run with family plan: ['account_takeover', 'social_engineering', 'trusted_device']
[seed=20260821] round 1/3 starting
[seed=20260821] round 1 Agent1.research complete in 9.67s
[seed=20260821] round 1 Agent2.specify complete in 30.46s
[seed=20260821] round 1 train/unseen generation complete in 0.00s
[seed=20260821] round 1 Agent3.analyze complete in 52.05s
[seed=20260821] round 1/3 complete | family=account_takeover | f1=0.4286 | novelty=0.8846
[seed=20260821] round 2/3 starting
[seed=20260821] round 2 Agent1.recommend_family complete in 6.11s
[seed=20260821] round 2 Agent1.research complete in 8.39s
[seed=20260821] round 2 Agent2.specify complete in 32.74s
[seed=20260821] round 2 train/unseen generation complete in 0.00s
[seed=20260821] round 2 Agent3.analyze complete in 31.22s
[seed=20260821] round 2/3 complete | family=account_takeover | f1=0.1250 | novelty=0.8956
[seed=20260821] round 3/3 starting
[seed=20260821] round 3 Agent1.recommend_family compl

## 5-Round Metrics Trend Gate

Before spending the full 20-round Kaggle GPU budget, run a short 5-round trend check using the
exact same config as the full run (Qwen agents, CTGAN generator, continual detector). Inspect the
printed per-round metrics below: F1/recall/precision/ROC-AUC should trend upward (or at least not
be flat), FPR should trend downward, novelty should decay from its round-1 value, and fidelity
should trend upward. Only proceed to the full 20-round run cells further down once this trend looks
reasonable. If it doesn't, stop here, adjust the config in the cell above, and re-run this check.


In [ ]:
from mastercard_defence.synthetic import build_metrics_dump

trend_check_config = load_config(str(project_dir / "config" / "default.yaml"))
trend_check_config["paths"]["memory_db"] = tempfile.mktemp(suffix="_adaptive_v2_trend_check.sqlite")
trend_check_config["seed"] = 20260822
trend_check_config["generator_backend"] = config["generator_backend"]
trend_check_config["generator_epochs"] = config["generator_epochs"]
trend_check_config["generator_training_attack_size"] = config["generator_training_attack_size"]
trend_check_config["generator_training_reference_size"] = config["generator_training_reference_size"]
trend_check_config["detector_mode"] = config["detector_mode"]
trend_check_config["detector_retrain_every"] = config["detector_retrain_every"]
trend_check_config["max_historical_pool"] = config["max_historical_pool"]
trend_check_config["max_replay_examples"] = config["max_replay_examples"]
trend_check_config["weakness_weight_multiplier"] = config["weakness_weight_multiplier"]
trend_check_config["model"]["context_size"] = config["model"]["context_size"]
trend_check_config["model"]["max_output_tokens"] = config["model"]["max_output_tokens"]
trend_check_config["pipeline"]["rounds"] = 5
trend_check_config["pipeline"]["synthetic_transactions"] = config["pipeline"]["synthetic_transactions"]
trend_check_config["pipeline"]["max_generated_attacks"] = config["pipeline"]["max_generated_attacks"]
trend_check_config["pipeline"]["fraud_rate"] = config["pipeline"]["fraud_rate"]

# Reuse the already-loaded `agents` (and its GGUF model) instead of creating a second SharedLocalLLM.
trend_check_loop = ClosedLoop(trend_check_config, agents=agents)
try:
    # run_robustness_suite (not run()) is required here: it is the only entry point that builds
    # and fits the CTGAN generator when generator_backend="ctgan". Calling run() directly falls
    # back to the procedural generator, silently skipping CTGAN entirely.
    trend_check_suite = trend_check_loop.run_robustness_suite(seeds=1, rounds=5)
finally:
    trend_check_loop.close()

trend_check_results = [result for run in trend_check_suite["by_seed"] for result in run["results"]]
trend_check_dump = build_metrics_dump(trend_check_results)
trend_columns = [
    "round", "attack_family", "f1", "recall", "precision", "roc_auc",
    "false_positive_rate", "attack_novelty_score",
    "attack_fidelity_behavioural_plausibility", "historical_robustness_f1",
]
print("TREND_CHECK_ROWS", len(trend_check_dump))
for row in trend_check_dump:
    print({column: row.get(column) for column in trend_columns})

print("\nReview the trend above: STOP and adjust config if F1/recall/precision/ROC-AUC are flat,")
print("FPR is rising, novelty isn't decaying from round 1, or fidelity isn't trending up.")
print("Only proceed to the full 20-round run cells below if this 5-round trend looks reasonable.")


[robustness] preparing CTGAN training corpus
[robustness] fitting CTGAN: rows=680 epochs=50


/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


[robustness] CTGAN ready in 11.58s; entering round loop
[robustness] starting suite: seeds=1, rounds=5
[robustness] starting seed 20260822 with family plan: ['account_takeover', 'cross_channel_anomaly', 'social_engineering', 'beneficiary_manipulation', 'low_and_slow']
[seed=20260822] starting 5-round run with family plan: ['account_takeover', 'cross_channel_anomaly', 'social_engineering', 'beneficiary_manipulation', 'low_and_slow']
[seed=20260822] round 1/5 starting
[seed=20260822] round 1 Agent1.research complete in 10.28s
[seed=20260822] round 1 Agent2.specify complete in 32.16s
[ctgan] family='account_takeover' attempt=3/32 matched=60/80
[ctgan] family='account_takeover' attempt=4/32 matched=67/80
[ctgan] family='account_takeover' attempt=5/32 matched=72/80
[ctgan] family='account_takeover' attempt=6/32 matched=76/80
[ctgan] family='account_takeover' attempt=3/32 matched=58/80
[ctgan] family='account_takeover' attempt=4/32 matched=67/80
[ctgan] family='account_takeover' attempt=5/32

In [8]:
import contextlib
import io

class Tee(io.TextIOBase):
    def __init__(self, *streams):
        self.streams = streams
    def write(self, text):
        for stream in self.streams:
            stream.write(text)
            stream.flush()
        return len(text)
    def flush(self):
        for stream in self.streams:
            stream.flush()

artifacts_dir = project_dir / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)
log_path = artifacts_dir / f"adaptive_v2_run_{run_stamp}.log"

with log_path.open("w", encoding="utf-8") as log_file, contextlib.redirect_stdout(Tee(sys.stdout, log_file)), contextlib.redirect_stderr(Tee(sys.stderr, log_file)):
    print(f"LOG_SAVED {log_path}")
    print(f"RUN_CONFIGURATION seeds=1 rounds={RUN_ROUNDS} agent_backend=QwenAgents generator=conditional_ctgan detector={config['detector_mode']} fraud_rate={config['pipeline']['fraud_rate']}")
    try:
        suite = loop.run_robustness_suite(seeds=1, rounds=RUN_ROUNDS)
    finally:
        loop.close()

print("One-seed adaptive v2 run completed")
print("Agent backend: QwenAgents")
print("Log:", log_path)

LOG_SAVED /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_run_20260828T143736Z.log
RUN_CONFIGURATION seeds=1 rounds=20 agent_backend=QwenAgents generator=conditional_ctgan detector=continual fraud_rate=0.02
[robustness] preparing CTGAN training corpus
[robustness] fitting CTGAN: rows=680 epochs=3


/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[robustness] CTGAN ready in 11.36s; entering round loop
[robustness] starting suite: seeds=1, rounds=20
[robustness] starting seed 20260821 with family plan: ['account_takeover', 'social_engineering', 'trusted_device', 'beneficiary_manipulation', 'cross_channel_anomaly', 'merchant_abuse', 'low_and_slow', 'account_takeover', 'social_engineering', 'trusted_device', 'beneficiary_manipulation', 'cross_channel_anomaly', 'merchant_abuse', 'low_and_slow', 'account_takeover', 'social_engineering', 'trusted_device', 'beneficiary_manipulation', 'cross_channel_anomaly', 'merchant_abuse']
[seed=20260821] starting 20-round run with family plan: ['account_takeover', 'social_engineering', 'trusted_device', 'beneficiary_manipulation', 'cross_channel_anomaly', 'merchant_abuse', 'low_and_slow', 'account_takeover', 'social_engineering', 'trusted_device', 'beneficiary_manipulation', 'cross_channel_anomaly', 'merchant_abuse', 'low_and_slow', 'account_takeover', 'social_engineering', 'trusted_device', 'bene

In [9]:
def to_jsonable(value):
    if hasattr(value, "model_dump"):
        return to_jsonable(value.model_dump())
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    return value

artifact = {
    "run_timestamp_utc": run_stamp,
    "experiment": "adaptive_v2_qwen_ctgan_static_detector",
    "agent_backend": type(agents).__name__,
    "generator_backend": "conditional_ctgan",
    "detector_mode": config["detector_mode"],
    "fraud_rate_target": config["pipeline"]["fraud_rate"],
    "seed_count": suite["seed_count"],
    "rounds": suite["rounds"],
    "summary": suite["summary"],
    "family_analysis": suite.get("family_analysis", []),
    "detector_version_analysis": suite.get("detector_version_analysis", []),
    "by_seed": suite["by_seed"],
}
result_path = artifacts_dir / f"adaptive_v2_results_{run_stamp}.json"
result_path.write_text(json.dumps(to_jsonable(artifact), indent=2), encoding="utf-8")

print("RESULTS_SAVED", result_path)
print("LOG_SAVED", log_path)
print("FAMILY_ANALYSIS_ROWS", len(artifact["family_analysis"]))
print("DETECTOR_VERSION_ANALYSIS", artifact["detector_version_analysis"])
print("ADAPTIVE_V2_KAGGLE_RUN_OK", suite["seed_count"], suite["rounds"])

RESULTS_SAVED /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_results_20260828T143736Z.json
LOG_SAVED /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_run_20260828T143736Z.log
FAMILY_ANALYSIS_ROWS 5
DETECTOR_VERSION_ANALYSIS [{'detector_version': 1, 'round_count': 2, 'precision': 0.7931, 'recall': 0.5, 'f1': 0.6122, 'roc_auc': 0.9565}, {'detector_version': 2, 'round_count': 2, 'precision': 0.8282, 'recall': 0.5938, 'f1': 0.6904, 'roc_auc': 0.9649}, {'detector_version': 3, 'round_count': 2, 'precision': 0.7573, 'recall': 0.5687, 'f1': 0.6485, 'roc_auc': 0.9543}, {'detector_version': 4, 'round_count': 2, 'precision': 0.8026, 'recall': 0.5875, 'f1': 0.6783, 'roc_auc': 0.9532}, {'detector_version': 5, 'round_count': 2, 'precision': 0.8167, 'recall': 0.5813, 'f1': 0.6783, 'roc_auc': 0.9473}, {'detector_version': 6, 'round_count': 2, 'precision': 0.7863, 'recall': 0.5813, 'f1': 0.6664, 'roc_auc': 0.959}, {'detector_version': 7, 'round_count': 2, 'precision': 0.7636, 'rec

In [10]:
from mastercard_defence.synthetic import build_metrics_dump

flattened_results = [result for run in suite["by_seed"] for result in run["results"]]
metrics_dump = build_metrics_dump(flattened_results)
metrics_dump_path = artifacts_dir / f"adaptive_v2_metrics_dump_{run_stamp}.json"
metrics_dump_path.write_text(json.dumps(metrics_dump, indent=2), encoding="utf-8")

print("METRICS_DUMP_SAVED", metrics_dump_path)
print("METRICS_DUMP_ROWS", len(metrics_dump))
print("METRICS_DUMP_JSON_START")
print(json.dumps(metrics_dump, indent=2))
print("METRICS_DUMP_JSON_END")

METRICS_DUMP_SAVED /kaggle/working/mastercard_hackathon/artifacts/adaptive_v2_metrics_dump_20260828T143736Z.json
METRICS_DUMP_ROWS 20
METRICS_DUMP_JSON_START
[
  {
    "round": 1,
    "attack_family": "account_takeover",
    "detector_version": 1,
    "attack_diversity_channel_entropy": 0.9834,
    "attack_diversity_unique_row_ratio": 1.0,
    "attack_fidelity_behavioural_plausibility": 0.5755,
    "attack_novelty_score": 1.0,
    "attack_difficulty_score": 0.5375,
    "family_coverage_cumulative_ratio": 0.1429,
    "family_coverage_cumulative_count": 1,
    "variant_redundancy_ratio": 0.0,
    "variant_unique_ratio": 1.0,
    "precision": 0.8043478260869565,
    "recall": 0.4625,
    "f1": 0.5873015873015873,
    "roc_auc": 0.9519164540816326,
    "false_positive_rate": 0.002295918367346939,
    "unseen_attack_evaluation_protocol": "unseen_attack_rows_and_legitimate_holdout",
    "historical_robustness_insufficient": true,
    "historical_robustness_precision": 0.0,
    "historical_ro